# Car Fuel Efficiency Analysis (1970–1982)

**Course:** 2323 CIS — Introduction to Data Science · King Khalid University
**Labs:** 14–15 — Lab project, GitHub publishing and presentation

## 1. Data question

> **How did car fuel efficiency change between 1970 and 1982, and which car characteristics (weight, cylinders, horsepower, region of origin) are most strongly linked to it?**

**Why it matters:** the 1973 and 1979 oil crises pushed fuel prices up sharply. This dataset lets us see whether cars actually became more efficient afterwards, and what made a car efficient.

**Notebook outline**

1. Data question
2. Setup and data loading
3. Data understanding
4. Data preparation
5. Visualizations and interpretation
6. Final analysis-ready dataset
7. Conclusions and limitations

## 2. Setup and data loading

All libraries below are pre-installed in Google Colab. Paths are relative, so the notebook runs the same way in Colab or locally.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)

# Save outputs next to the repo root whether we run from the root or from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = ROOT / "outputs"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)
print("Outputs will be saved to:", OUT.resolve())

The **Auto MPG** dataset (UCI Machine Learning Repository) is loaded directly from the internet with `seaborn.load_dataset`. Nothing is typed in by hand.

In [ ]:
df_raw = sns.load_dataset("mpg")
df_raw.head()

## 3. Data understanding

### 3.1 Shape and columns

In [ ]:
print("Rows:", df_raw.shape[0])
print("Columns:", df_raw.shape[1])
print("Column names:", list(df_raw.columns))

### 3.2 Data types

In [ ]:
df_raw.dtypes.to_frame("data type")

### 3.3 Missing values

In [ ]:
missing = df_raw.isna().sum().to_frame("missing")
missing["percent"] = (missing["missing"] / len(df_raw) * 100).round(2)
missing[missing["missing"] > 0]

### 3.4 Descriptive statistics

In [ ]:
df_raw.describe().round(2)

### 3.5 Categories and duplicates

In [ ]:
print("Cars per region:")
print(df_raw["origin"].value_counts(), "\n")
print("Cars per number of cylinders:")
print(df_raw["cylinders"].value_counts().sort_index(), "\n")
print("Duplicate rows:", df_raw.duplicated().sum())

### 3.6 A hidden problem: messy brand names

The `name` column combines brand and model in one string (for example `chevrolet chevelle malibu`). Looking at the first word shows misspellings and abbreviations of the same brand.

In [ ]:
first_word = df_raw["name"].str.split().str[0]
suspicious = ["chevrolet", "chevroelt", "chevy", "volkswagen", "vokswagen", "vw",
              "toyota", "toyouta", "mazda", "maxda", "mercedes", "mercedes-benz"]
first_word[first_word.isin(suspicious)].value_counts().sort_index()

**Data-quality diagnosis**

| Issue | Where | Planned fix |
| --- | --- | --- |
| Missing values | `horsepower` | Fill with the median horsepower of cars with the same number of cylinders |
| Brand hidden inside text | `name` | Extract the brand with a regular expression |
| Misspelled / abbreviated brands | `name` | Map spelling variants to one standard brand |
| Imperial units | `mpg`, `weight`, `displacement` | Add metric columns (L/100 km, kg, litres) |
| Year stored as two digits | `model_year` | Convert to a full year (70 → 1970) |

## 4. Data preparation

We work on a copy so that the raw data stays unchanged.

### 4.1 Cleaning: fill missing horsepower

**Decision:** engine power depends strongly on the number of cylinders, so each missing value is filled with the **median horsepower of cars with the same number of cylinders**. Using the median rather than the mean means a few very powerful cars don't pull the estimate up. Dropping the rows would also remove their `mpg` values, which we need.

In [ ]:
df = df_raw.copy()

print("Rows with missing horsepower:")
display(df.loc[df["horsepower"].isna(), ["name", "cylinders", "displacement", "horsepower"]])

df["horsepower"] = df["horsepower"].fillna(
    df.groupby("cylinders")["horsepower"].transform("median")
)
print("Missing horsepower after filling:", df["horsepower"].isna().sum())

### 4.2 Regular-expression extraction: brand and model

**Decision:** the regex `^([a-z\-]+)` captures the first word of the name (letters and hyphens) as the brand, and `^\S+\s+(.*)$` captures everything after it as the model. A small dictionary then merges spelling variants into one brand.

In [ ]:
df["name"] = df["name"].str.strip().str.lower()
df["brand_raw"] = df["name"].str.extract(r"^([a-z\-]+)", expand=False)
df["model"] = df["name"].str.extract(r"^\S+\s+(.*)$", expand=False)

brand_fixes = {
    "chevroelt": "chevrolet", "chevy": "chevrolet",
    "vokswagen": "volkswagen", "vw": "volkswagen",
    "toyouta": "toyota", "maxda": "mazda",
    "mercedes-benz": "mercedes", "capri": "mercury",
    "hi": "international",
}
df["brand"] = df["brand_raw"].replace(brand_fixes)

changed = df.loc[df["brand_raw"] != df["brand"], ["name", "brand_raw", "brand"]]
print(f"Brand names corrected: {len(changed)} rows")
print(f"Distinct brands before: {df['brand_raw'].nunique()}  ->  after: {df['brand'].nunique()}")
changed.drop_duplicates("brand_raw")

### 4.3 Transformation: metric units and readable categories

**Decision:** the audience in Saudi Arabia uses metric units, so we add:

- `year`: full model year (70 → 1970)
- `l_per_100km`: litres per 100 km = 235.215 ÷ mpg (lower is better)
- `weight_kg`: pounds × 0.453592
- `engine_litres`: cubic inches × 0.0163871
- `weight_class`: weight grouped into four bands for easier comparison
- `origin`: neater region labels (USA, Europe, Japan)

In [ ]:
df["year"] = 1900 + df["model_year"]
df["l_per_100km"] = (235.215 / df["mpg"]).round(2)
df["weight_kg"] = (df["weight"] * 0.453592).round(0)
df["engine_litres"] = (df["displacement"] * 0.0163871).round(2)
df["origin"] = df["origin"].map({"usa": "USA", "europe": "Europe", "japan": "Japan"})
df["weight_class"] = pd.cut(
    df["weight_kg"],
    bins=[0, 1000, 1400, 1800, np.inf],
    labels=["Light (<1000 kg)", "Medium (1000–1400 kg)", "Heavy (1400–1800 kg)", "Very heavy (>1800 kg)"],
)

df[["name", "brand", "year", "mpg", "l_per_100km", "weight_kg", "engine_litres", "weight_class", "origin"]].head()

### 4.4 Aggregation, reshaping and time series: yearly averages per region

**Decision:** to see change over time, we **group** cars by year and region, **aggregate** the average mpg and car count, then **reshape** (pivot) the long table into a wide table with one column per region.

In [ ]:
yearly = (
    df.groupby(["year", "origin"], as_index=False)
      .agg(avg_mpg=("mpg", "mean"), avg_weight_kg=("weight_kg", "mean"), cars=("mpg", "size"))
)

yearly_wide = yearly.pivot(index="year", columns="origin", values="avg_mpg").round(1)
yearly_wide["All regions"] = df.groupby("year")["mpg"].mean().round(1)
yearly_wide

## 5. Visualizations and interpretation

### 5.1 How did fuel efficiency change over time?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=yearly, x="year", y="avg_mpg", hue="origin", marker="o", ax=ax)

top = ax.get_ylim()[1]
for yr, label in [(1973, "1973 oil crisis"), (1979, "1979 oil crisis")]:
    ax.axvline(yr, color="grey", linestyle="--", linewidth=1)
    ax.text(yr + 0.15, top - 1, label, fontsize=9, color="grey", va="top")

ax.set(title="Average fuel efficiency by model year and region",
       xlabel="Model year", ylabel="Average fuel efficiency (miles per gallon)")
ax.legend(title="Region")
fig.tight_layout()
fig.savefig(FIG / "1_mpg_trend_by_region.png", dpi=150)
plt.show()

overall = df.groupby("year")["mpg"].mean()
y0, y1 = overall.index.min(), overall.index.max()
print(f"Average mpg rose from {overall[y0]:.1f} in {y0} to {overall[y1]:.1f} in {y1} "
      f"(+{(overall[y1] / overall[y0] - 1) * 100:.0f}%).")
print(f"In litres per 100 km: {235.215 / overall[y0]:.1f} -> {235.215 / overall[y1]:.1f}.")
print("Average mpg by region over the whole period:")
print(df.groupby("origin")["mpg"].mean().round(1).sort_values(ascending=False).to_string())

**Interpretation:**

- Fuel efficiency improved strongly between 1970 and 1982. The printed figures above give the exact size of the change.
- **American cars** show the clearest trend: after the 1973 oil crisis their average rose almost every year, roughly doubling from about 15 to about 29.5 mpg.
- **Japanese and European cars** were more efficient than American cars in every year. Their lines are more jagged because each year contains only a few European or Japanese models, so one unusual car moves the average a lot (the `cars` column in the yearly table shows the counts).

### 5.2 Which characteristic is most strongly linked to efficiency?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
sns.scatterplot(data=df, x="weight_kg", y="mpg", hue="origin", style="origin", alpha=0.75, ax=ax)
sns.regplot(data=df, x="weight_kg", y="mpg", scatter=False, order=2, ci=None,
            color="black", line_kws={"linewidth": 1.2}, ax=ax)
ax.set(title="Heavier cars use more fuel",
       xlabel="Car weight (kg)", ylabel="Fuel efficiency (miles per gallon)")
ax.legend(title="Region")
fig.tight_layout()
fig.savefig(FIG / "2_weight_vs_mpg.png", dpi=150)
plt.show()

corr = (df[["mpg", "weight_kg", "horsepower", "engine_litres", "cylinders", "acceleration", "year"]]
        .corr()["mpg"].drop("mpg").sort_values())
print("Correlation of each characteristic with mpg (-1 = strong negative, +1 = strong positive):")
print(corr.round(2).to_string())
print(f"\nStrongest link: {corr.abs().idxmax()} (r = {corr[corr.abs().idxmax()]:.2f})")

**Interpretation:**

- The points curve clearly downward: heavier cars travel fewer miles per gallon. The correlation list confirms that weight, engine size, horsepower and cylinders all have strong negative links with mpg. These four rise together, because bigger cars tend to have bigger engines.
- **Weight also explains the regional gap from 5.1.** Almost all Japanese and European cars sit on the light, left side of the chart, while many American cars weigh 1,500–2,300 kg. American cars were less efficient largely because they were heavier.
- Newer model years have a positive link with mpg, which matches the trend in 5.1.

### 5.3 How does the number of cylinders affect efficiency?

In [ ]:
counts = df["cylinders"].value_counts().sort_index()
order = list(counts.index)

fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=df, x="cylinders", y="mpg", order=order, color="#7fb3d5", ax=ax)
ax.set_xticks(range(len(order)))
ax.set_xticklabels([f"{c}\n(n={counts[c]})" for c in order])
ax.set(title="Fuel efficiency by number of engine cylinders",
       xlabel="Number of cylinders (number of cars)", ylabel="Fuel efficiency (miles per gallon)")
fig.tight_layout()
fig.savefig(FIG / "3_mpg_by_cylinders.png", dpi=150)
plt.show()

print("Median mpg by cylinders:")
print(df.groupby("cylinders")["mpg"].median().round(1).to_string())

**Interpretation:** 4-cylinder cars have the highest typical efficiency, and 8-cylinder cars have the lowest. Each step up from 4 to 6 to 8 cylinders lowers the median mpg. The 3- and 5-cylinder groups contain only a handful of cars (see *n* on the axis), so no firm conclusion should be drawn about them.

## 6. Final analysis-ready dataset

The clean dataset keeps the original measurements plus the new columns, and is saved to `outputs/mpg_clean.csv`. The yearly summary table is saved to `outputs/yearly_mpg_by_region.csv`.

In [ ]:
final_cols = ["brand", "model", "year", "origin", "cylinders", "engine_litres", "horsepower",
              "weight_kg", "weight_class", "acceleration", "mpg", "l_per_100km"]
mpg_clean = df[final_cols].sort_values(["year", "brand"]).reset_index(drop=True)

mpg_clean.to_csv(OUT / "mpg_clean.csv", index=False)
yearly_wide.to_csv(OUT / "yearly_mpg_by_region.csv")

print("Final dataset shape:", mpg_clean.shape)
print("Missing values remaining:", int(mpg_clean.isna().sum().sum()))
print("Saved files:")
for f in sorted(OUT.rglob("*.*")):
    print("  ", f.relative_to(ROOT))
mpg_clean.head()

## 7. Conclusions and limitations

**Answer to the data question**

1. Car fuel efficiency rose sharply between 1970 and 1982, especially after the 1973 and 1979 oil crises (see 5.1).
2. Japanese and European cars were more efficient than American cars in every year, but American cars improved the most steadily, roughly doubling their average mpg (see 5.1).
3. Weight is the characteristic most strongly linked to efficiency: heavier cars, with larger engines and more cylinders, use more fuel (see 5.2 and 5.3).
4. The regional gap is largely a weight gap: Japanese and European cars were much lighter than American cars (see 5.2).

**Limitations**

- **Old, narrow data:** 398 car models sold in the USA from 1970 to 1982. The results may not apply to modern cars or to the Saudi market.
- **Models, not sales:** each model counts once however many were sold, so the averages are not market-wide averages.
- **Imputed values:** a few missing horsepower values were filled with group medians, which is an estimate.
- **Correlation, not causation:** weight, engine size and cylinders rise together, so this analysis cannot separate their individual effects.

**Reproducibility:** restart the runtime and choose *Run all*. Every number and chart above is produced by the code in this notebook.